# SymboLM (v1.2) — Stage 4: Symbolic Efficiency & Hard EOS Alignment

**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Exoskeleton:** Antigravity (Google DeepMind)  
**Hardware:** Kaggle Dual NVIDIA Tesla T4 GPUs (32 GB VRAM)  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

---  
### Core Objectives:
1. **Monotonic Length Economy:** Penalize reasoning bloat beyond optimal theorem proof length.
2. **Eliminate Looping Repetition:** Impose -2.5 penalty on repeated answer strings (`527 527...`, `97 97...`).
3. **Hard EOS Cutoff (+1.2):** Reward immediate emission of `<｜end of sentence｜>` within 4 tokens of answer verification.
4. **Purge Base Web Debris:** Penalize lingering base Qwen vocabulary tokens (`endphp`, `IconModule`, etc.).

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Install required libraries & resolve package conflicts
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft trl accelerate sympy datasets bitsandbytes huggingface_hub


In [ ]:
# Cell 3: Clone latest SymboLM repository
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
print('[OK] Successfully cloned latest SymboLM repository!')


In [ ]:
# Cell 4: Mount Stage 3.5 Concept GRPO Policy Adapter as warm-start base
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/stage3_concept_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for Stage 3 adapter in mounted kernel sources
for p in Path('/kaggle/input').glob('**/stage3_adapter'):
    if (p / 'adapter_config.json').exists():
        print(f'[OK] Found Stage 3 adapter at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# 2. Search for stage 3 tarball archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*stage3*policy*.tar.gz'):
        print(f'[OK] Found Stage 3 tar archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

# 3. Fallback: Download from Hugging Face Hub
if not found or not (target_dir / 'adapter_model.safetensors').exists():
    print('[INFO] Pulling Stage 3.5 adapter directly from Hugging Face Hub...')
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='gyawalisanish0/symboLM-checkpoints',
        allow_patterns='stage3_concept_adapter/*',
        local_dir='/kaggle/working/hf_download'
    )
    src_hf = Path('/kaggle/working/hf_download/stage3_concept_adapter')
    for item in src_hf.iterdir():
        if item.is_file():
            shutil.copy2(item, target_dir / item.name)

print('[Status] Validating Warm-Start Adapter Files:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')

assert (target_dir / 'adapter_config.json').exists(), 'CRITICAL: adapter_config.json missing!'
assert (target_dir / 'adapter_model.safetensors').exists(), 'CRITICAL: adapter_model.safetensors missing!'
print('[VERIFIED] Stage 3.5 Concept GRPO Policy Adapter is ready for Stage 4 optimization!')


In [ ]:
# Cell 5: Run Stage 4 Efficiency & Hard EOS GRPO Training
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/symboLM/training/stage4_efficiency_train.py \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --warmstart_adapter /kaggle/working/symboLM/checkpoints/stage3_concept_adapter \
  --output_dir /kaggle/working/stage4_adapter \
  --data_file /kaggle/working/symboLM/data/stage3_unified_curriculum.jsonl \
  --max_steps 150 \
  --max_completion_length 128 \
  --num_generations 4 \
  --learning_rate 1e-5


In [ ]:
# Cell 6: Post-Training Single-Step EOS Audit
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_name = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
adapter_dir = '/kaggle/working/stage4_adapter'
if not (Path(adapter_dir) / 'adapter_config.json').exists():
    print('[Warning] adapter_config.json not found at', adapter_dir)
else:
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True
    )
    model.resize_token_embeddings(151712)
    model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    test_prompts = [
        'If x + 1/x = 5, find the exact value of x^4 + 1/x^4.',
        'Find the highest power of 3 that divides 200!.',
        'What is the chemical symbol for Potassium?',
        'How do I tell my direct report that their deadline is slipping without discouraging them?'
    ]
    print('=== STAGE 4 POST-TRAINING AUDIT (Verifying Single-Step EOS & Zero Debris) ===')
    for q in test_prompts:
        prompt_str = f'<｜User｜>{q}<｜Assistant｜><think>\n'
        inp = tokenizer(prompt_str, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=100, temperature=0.2, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=False)
        tok_count = len(out[0][inp['input_ids'].shape[1]:])
        print(f'\n[PROMPT] {q}')
        print(f'[TOKENS EMITTED] {tok_count}')
        print(f'[GENERATION]\n{gen}')


In [ ]:
# Cell 7: Archive Stage 4 Final Policy Adapter
!tar -czf /kaggle/working/symboLM_stage4_final_policy.tar.gz -C /kaggle/working/stage4_adapter .
print('[COMPLETE] Successfully archived symboLM_stage4_final_policy.tar.gz!')
